# E3 — Functional stitching: does the receiving network *use* what the map delivers?

A3 and B2 measured whether a linear map **preserves information**: held-out
R² of 0.737 across LLM layers, 0.592 into SigLIP's space. This notebook
asks the strictly stronger question: **can the receiving network consume
it?**

The two can diverge. A map can preserve variance in directions the next
stage ignores while destroying the ones it depends on. R² grades the
translation; only running the downstream stage grades whether the
translation is *usable*.

**Why the vision pair and not the LLM pair.** Stitching GPT-2 into Pythia
runs into a vocabulary problem — Pythia's LM head predicts over its own
tokenizer, so perplexity is not directly comparable. The vision pair has no
such obstacle: SigLIP's text tower is the downstream consumer, it is
unchanged, and retrieval is the functional score.

**The three arms.**

| Arm | Path | What it isolates |
|---|---|---|
| CEILING | SigLIP image tower -> SigLIP text tower | the intact system |
| STITCHED | MobileCLIP -> W -> SigLIP text tower | does the consumer accept a translated representation? |
| RAW | MobileCLIP zero-padded -> SigLIP text tower | the floor: no map |

**Pre-registered reading** (fixed before running). Define the stitching
penalty as `1 - R@5(stitched) / R@5(ceiling)`:

| Penalty | Interpretation |
|---|---|
| < 0.10 | the consumer accepts the translation almost transparently |
| 0.10-0.30 | usable with measurable degradation - report the number |
| > 0.30 | information survives the map (R² 0.592) but the consumer cannot exploit it - the interesting negative |

From B3 the expected penalty is about 0.05, because R@5 already reached
95.1% of ceiling. The value of running it explicitly is that it separates
*preserved* from *usable*, and it sets up the layer sweep in section 4,
where the answer is genuinely unknown.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ["DATA_DIR"] = "/content/drive/MyDrive/convergence_experiment"

In [ ]:
import numpy as np
from pathlib import Path

DATA_DIR = Path(os.environ["DATA_DIR"])
pairs = np.load(str(DATA_DIR / "pairs.npz"))
ad    = np.load(str(DATA_DIR / "adapter.npz"))
te, W = ad["eval_idx"], ad["W_ridge"].astype(np.float64)

def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)

def recall(S):
    order = np.argsort(-S, axis=1)
    r = (order == np.arange(len(S))[:, None]).argmax(1)
    return {k: float((r < k).mean()) for k in (1, 5, 10)}

sig = l2n(pairs["sig_img"][te].astype(np.float64))
txt = l2n(pairs["sig_txt"][te].astype(np.float64))
mob = pairs["mob_img"][te].astype(np.float64)
stitched = l2n(mob @ W)
raw = np.zeros_like(sig); raw[:, :mob.shape[1]] = mob; raw = l2n(raw)
print(f"{len(te)} held-out images; the SigLIP text tower is the consumer "
      "and is unchanged in every arm")

## 1. The three arms, scored by the consumer

In [ ]:
arms = {"CEILING  (SigLIP native)": sig,
        "STITCHED (MobileCLIP -> W)": stitched,
        "RAW      (no map)": raw}
res = {}
for name, gal in arms.items():
    res[name] = recall(txt @ gal.T)
    print(f"{name:28s} " +
          "  ".join(f"R@{k}={v:.3f}" for k, v in res[name].items()))

c = res["CEILING  (SigLIP native)"]
s = res["STITCHED (MobileCLIP -> W)"]
print("\nstitching penalty (1 - stitched/ceiling):")
for k in (1, 5, 10):
    pen = 1 - s[k] / c[k]
    print(f"  R@{k:<2d}  {pen:+.3f}   ({100*s[k]/c[k]:.1f}% of ceiling)")

pen5 = 1 - s[5] / c[5]
print("\nVERDICT:", 
      "consumer accepts the translation almost transparently" if pen5 < 0.10
      else ("usable with measurable degradation" if pen5 <= 0.30
            else "information survives the map but the consumer cannot "
                 "exploit it"))

## 2. Preserved vs usable — the distinction this notebook exists for

R² and cosine grade the *translation*. Recall grades what the *consumer*
does with it. Printing them together shows how far apart the two questions
can sit.

In [ ]:
r2 = 1 - ((pairs["sig_img"][te] - mob @ W) ** 2).sum() / \
         ((pairs["sig_img"][te] - pairs["sig_img"][te].mean(0)) ** 2).sum()
cos = float((stitched * sig).sum(1).mean())
print(f"PRESERVED : held-out R2 {r2:.3f}   mean cosine to target {cos:.3f}")
print(f"USABLE    : R@5 {s[5]:.3f} vs ceiling {c[5]:.3f} "
      f"= {100*s[5]/c[5]:.1f}% of ceiling")
print(f"FLOOR     : R@5 {res['RAW      (no map)'][5]:.3f} "
      "(the same vectors, no map)")
print("\nR2 of 0.592 sounds mediocre and R@5 at 95% of ceiling sounds "
      "excellent.")
print("Both are correct: R2 penalizes magnitude, the consumer sees only "
      "angle.")

## 3. Does the consumer need a *good* map, or just any map?

A control the earlier stages did not run: fit W on progressively fewer
training pairs and watch where the consumer's performance falls off. If
retrieval degrades gracefully, the consumer tolerates an approximate
translation; if it falls off a cliff, the map must be accurate to be
usable at all.

In [ ]:
tr_all = ad["train_idx"]
X = pairs["mob_img"][tr_all].astype(np.float64)
Y = pairs["sig_img"][tr_all].astype(np.float64)
d = X.shape[1]

print(f"{'n_train':>8s} {'rows/dim':>9s} {'R2':>7s} {'cos':>7s} "
      f"{'R@5':>7s} {'% ceiling':>10s}")
for n in (200, 400, 800, 1500, 3000):
    if n > len(tr_all):
        continue
    Wn = np.linalg.solve(X[:n].T @ X[:n] + 1e-2 * np.eye(d), X[:n].T @ Y[:n])
    Pn = mob @ Wn
    r2n = 1 - ((pairs["sig_img"][te] - Pn) ** 2).sum() / \
              ((pairs["sig_img"][te] -
                pairs["sig_img"][te].mean(0)) ** 2).sum()
    gn = l2n(Pn)
    rn = recall(txt @ gn.T)
    print(f"{n:8d} {n/d:9.1f} {r2n:7.3f} "
          f"{float((gn*sig).sum(1).mean()):7.3f} {rn[5]:7.3f} "
          f"{100*rn[5]/c[5]:9.1f}%")
print("\nwatch the rows/dim column against the >=5 rule: the point where")
print("R2 turns pessimistic is the point where the map is starved, not")
print("the point where the consumer stops tolerating it")

## 4. The open question: how deep can the splice go?

Sections 1-3 stitch at the *output* of the image tower, which is where the
adapter already lives. The genuinely unknown case is splicing **inside**
the network: run an image through MobileCLIP's early blocks, map those
intermediate activations into SigLIP's residual stream, and let SigLIP's
remaining blocks finish the forward pass.

That is harder for two reasons this project has already measured. The map
must land in the distribution the next block expects — scale, offset, and
the statistics layer norm was calibrated against — and layer
correspondence is soft at the extremes (report §C.7: sharp through the
middle depths, scattered at the embedding and output layers). The
prediction that follows: mid-depth splices should work, and splices at the
first or last block should not.

The cell below is a scaffold, not a result: it lays out the procedure and
the controls, and is left unrun deliberately, in the same spirit as every
other pre-registration in this project.

In [ ]:
# SCAFFOLD - not run. The procedure, so the next person does not re-derive it.
#
# 1. Extract intermediate activations from both towers on the same images:
#       mob_h[l] = MobileCLIP stage-l output, pooled to [N, d_l]
#       sig_h[m] = SigLIP block-m residual stream, pooled to [N, 768]
#    (both towers expose hidden states; SigLIP via output_hidden_states=True)
#
# 2. For each candidate pair (l, m), fit ridge mob_h[l] -> sig_h[m] on the
#    training split, exactly as A3 does, and record held-out R2. This gives
#    the rectangular correspondence matrix, read by row argmax rather than
#    by an assumed proportional mapping (report §C.7).
#
# 3. FUNCTIONAL test for the best (l, m): replace SigLIP's block-m input
#    with the mapped activation and run blocks m+1..end plus the projection
#    head. Score retrieval against the unchanged text tower.
#
# 4. Controls, all necessary:
#       - ceiling  : intact SigLIP
#       - shuffled : the same map applied to mismatched rows
#       - identity : mapped activation replaced by SigLIP's own (should
#                    reproduce the ceiling exactly - a wiring check)
#       - depth    : repeat at first, middle and last blocks
#
# 5. Distribution matching is the likely failure mode: standardize the
#    mapped activation to the receiving block's own per-dimension mean and
#    variance before injection, and report results with and without it.
print("scaffold only - see the markdown above for the procedure")

## 5. What E3 adds

- **Preserved is not usable.** R² and cosine grade the map; only the
  downstream consumer grades whether the map can be *used*. This notebook
  measures both on the same vectors and reports the gap.
- **The penalty is a number, not an impression.** Reporting `1 -
  stitched/ceiling` makes the cost of interoperability explicit and
  comparable across future encoder swaps.
- **The data-efficiency curve** shows whether the consumer needs an
  accurate map or merely an approximate one - and separates "the map is
  starved" from "the consumer is intolerant", which the rows-per-dimension
  column makes visible.
- **The depth question is left open and specified**, with its prediction
  derived from §C.7 rather than invented afterwards.